# ⚙️ Setup Planilha NFP — Calendário de Comunicação

Este notebook configura automaticamente:
- ✅ As 3 abas da planilha Google Sheets (CAMPANHAS, CLIENTES, LINKS CALENDÁRIOS)
- ✅ O código Apps Script vinculado (calendário visual + emails automáticos)
- ✅ Publica o webapp do calendário
- ✅ Preenche os links na aba LINKS CALENDÁRIOS

**Como usar:**
1. Execute a célula 1 (instala dependências)
2. Preencha os campos na célula 2 com suas credenciais
3. Execute as células em ordem
4. Na célula 3, uma janela de autorização vai abrir — autorize com sua conta Google

In [ ]:
# ━━━ CÉLULA 1 — Instala dependências ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip install -q google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

In [ ]:
# ━━━ CÉLULA 2 — Configurações ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Cole aqui o conteúdo do client_secret.json (baixado do Google Cloud Console)
# Se não tiver: console.cloud.google.com → Credenciais → + Criar → OAuth 2.0 → Aplicativo de desktop

CLIENT_SECRET_JSON = '''{
  "installed": {
    "client_id": "COLE_AQUI_SEU_CLIENT_ID",
    "client_secret": "COLE_AQUI_SEU_CLIENT_SECRET",
    "redirect_uris": ["urn:ietf:wg:oauth:2.0:oob", "http://localhost"],
    "auth_uri": "https://accounts.google.com/o/oauth2/auth",
    "token_uri": "https://oauth2.googleapis.com/token"
  }
}'''

# ID da planilha (já criada)
SPREADSHEET_ID = "1icbio-EAFvZggaQEuVvth_GCAINGLyD5WpNuGSRGaXo"

# Clientes
CLIENTES = [
    {"slug": "desatadora",      "name": "Desatadora dos Nós",   "operador_email": "roseane.welter@timecaptacao.com.br",  "copy_email": "jinlee@timecaptacao.com.br",       "criacao_email": "vinicius.torres@timecaptacao.com.br", "approver_email": "roseane.welter@timecaptacao.com.br"},
    {"slug": "santa_dulce",     "name": "Santa Dulce",          "operador_email": "barbara.aquino@timecaptacao.com.br", "copy_email": "jinlee@timecaptacao.com.br",       "criacao_email": "vinicius.torres@timecaptacao.com.br", "approver_email": "barbara.aquino@timecaptacao.com.br"},
    {"slug": "santo_antonio",   "name": "Santo Antonio",        "operador_email": "roseane.welter@timecaptacao.com.br",  "copy_email": "jinlee@timecaptacao.com.br",       "criacao_email": "vinicius.torres@timecaptacao.com.br", "approver_email": "roseane.welter@timecaptacao.com.br"},
    {"slug": "beato_donizetti", "name": "Beato Donizetti",      "operador_email": "roseane.welter@timecaptacao.com.br",  "copy_email": "jinlee@timecaptacao.com.br",       "criacao_email": "vinicius.torres@timecaptacao.com.br", "approver_email": "roseane.welter@timecaptacao.com.br"},
    {"slug": "barco_hospital",  "name": "Barco Hospital",       "operador_email": "barbara.aquino@timecaptacao.com.br", "copy_email": "jinlee@timecaptacao.com.br",       "criacao_email": "vinicius.torres@timecaptacao.com.br", "approver_email": "barbara.aquino@timecaptacao.com.br"},
    {"slug": "afesu",           "name": "Afesu",                "operador_email": "natalia.valverde@timecaptacao.com.br","copy_email": "natalia.valverde@timecaptacao.com.br","criacao_email": "vinicius.torres@timecaptacao.com.br","approver_email": "natalia.valverde@timecaptacao.com.br"},
    {"slug": "amparo",          "name": "Amparo",               "operador_email": "barbara.aquino@timecaptacao.com.br", "copy_email": "barbara.aquino@timecaptacao.com.br","criacao_email": "vinicius.torres@timecaptacao.com.br", "approver_email": "barbara.aquino@timecaptacao.com.br"},
    {"slug": "encontro_cristo", "name": "Encontro com Cristo",  "operador_email": "barbara.aquino@timecaptacao.com.br", "copy_email": "jinlee@timecaptacao.com.br",       "criacao_email": "vinicius.torres@timecaptacao.com.br", "approver_email": "barbara.aquino@timecaptacao.com.br"},
]

print("✅ Configurações prontas")

In [ ]:
# ━━━ CÉLULA 3 — Autorização Google ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, os
from google_auth_oauthlib.flow import InstalledAppFlow

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/script.projects",
    "https://www.googleapis.com/auth/script.deployments",
    "https://www.googleapis.com/auth/gmail.send",
]

with open("/tmp/client_secret.json", "w") as f:
    f.write(CLIENT_SECRET_JSON)

flow = InstalledAppFlow.from_client_secrets_file("/tmp/client_secret.json", SCOPES)
creds = flow.run_console()  # Mostra um link — abra no navegador, autorize e cole o código aqui

with open("/tmp/google_token.json", "w") as f:
    f.write(creds.to_json())

print("\n✅ Token salvo!")
print("\n🔑 Refresh token (salve como secret GOOGLE_REFRESH_TOKEN no GitHub):")
print(json.loads(creds.to_json()).get("refresh_token", "(não encontrado)"))

In [ ]:
# ━━━ CÉLULA 4 — Configura as abas da planilha ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request

creds = Credentials.from_authorized_user_file("/tmp/google_token.json", SCOPES)
svc = build("sheets", "v4", credentials=creds)

def hex_rgb(h):
    h = h.lstrip("#")
    return {"red": int(h[0:2],16)/255, "green": int(h[2:4],16)/255, "blue": int(h[4:6],16)/255}

def header_fmt(bg="#2C3E50"):
    return {"backgroundColor": hex_rgb(bg), "textFormat": {"foregroundColor": {"red":1,"green":1,"blue":1}, "bold":True, "fontSize":11}, "verticalAlignment":"MIDDLE", "horizontalAlignment":"CENTER"}

def ensure_sheet(sid, title):
    meta = svc.spreadsheets().get(spreadsheetId=sid).execute()
    for sh in meta.get("sheets", []):
        if sh["properties"]["title"] == title:
            return sh["properties"]["sheetId"]
    r = svc.spreadsheets().batchUpdate(spreadsheetId=sid, body={"requests":[{"addSheet":{"properties":{"title":title}}}]}).execute()
    return r["replies"][0]["addSheet"]["properties"]["sheetId"]

# Remove aba padrão
meta = svc.spreadsheets().get(spreadsheetId=SPREADSHEET_ID).execute()
for sh in meta.get("sheets",[]):
    if sh["properties"]["title"] in ("Plan1","Sheet1","Página1"):
        svc.spreadsheets().batchUpdate(spreadsheetId=SPREADSHEET_ID, body={"requests":[{"deleteSheet":{"sheetId":sh["properties"]["sheetId"]}}]}).execute()
        print(f"Aba '{sh['properties']['title']}' removida")

id_camp = ensure_sheet(SPREADSHEET_ID, "CAMPANHAS")
id_cli  = ensure_sheet(SPREADSHEET_ID, "CLIENTES")
id_lnk  = ensure_sheet(SPREADSHEET_ID, "LINKS CALENDÁRIOS")

# --- CAMPANHAS ---
h_camp = ["Cliente","Tipo","Mês-Ano","Data Planejada","Campanha / Assunto","Copy","Arte","Dados","Status Geral","Link Drive (insumos)","Link HTML (RD Station)","Observações"]
widths = [100,110,90,120,260,60,60,60,180,200,200,200]
reqs = [
    {"updateSheetProperties":{"properties":{"sheetId":id_camp,"gridProperties":{"frozenRowCount":1,"frozenColumnCount":1}},"fields":"gridProperties.frozenRowCount,gridProperties.frozenColumnCount"}},
    {"updateDimensionProperties":{"range":{"sheetId":id_camp,"dimension":"ROWS","startIndex":0,"endIndex":1},"properties":{"pixelSize":36},"fields":"pixelSize"}},
    {"updateCells":{"range":{"sheetId":id_camp,"startRowIndex":0,"endRowIndex":1,"startColumnIndex":0,"endColumnIndex":len(h_camp)},"rows":[{"values":[{"userEnteredValue":{"stringValue":h},"userEnteredFormat":header_fmt()} for h in h_camp]}],"fields":"userEnteredValue,userEnteredFormat"}},
    {"setDataValidation":{"range":{"sheetId":id_camp,"startRowIndex":1,"endRowIndex":500,"startColumnIndex":1,"endColumnIndex":2},"rule":{"condition":{"type":"ONE_OF_LIST","values":[{"userEnteredValue":v} for v in ["email","instagram","whatsapp","sms","outros"]]},"showCustomUi":True,"strict":True}}},
    {"repeatCell":{"range":{"sheetId":id_camp,"startRowIndex":1,"endRowIndex":500,"startColumnIndex":5,"endColumnIndex":9},"cell":{"userEnteredFormat":{"horizontalAlignment":"CENTER"}},"fields":"userEnteredFormat.horizontalAlignment"}},
    {"repeatCell":{"range":{"sheetId":id_camp,"startRowIndex":1,"endRowIndex":500,"startColumnIndex":3,"endColumnIndex":4},"cell":{"userEnteredFormat":{"numberFormat":{"type":"DATE","pattern":"dd/MM/yyyy"}}},"fields":"userEnteredFormat.numberFormat"}},
    {"addConditionalFormatRule":{"rule":{"ranges":[{"sheetId":id_camp,"startRowIndex":1,"endRowIndex":500,"startColumnIndex":8,"endColumnIndex":9}],"booleanRule":{"condition":{"type":"TEXT_CONTAINS","values":[{"userEnteredValue":"Pronto"}]},"format":{"backgroundColor":hex_rgb("#D1FAE5"),"textFormat":{"foregroundColor":hex_rgb("#065F46"),"bold":True}}}},"index":0}},
    {"addConditionalFormatRule":{"rule":{"ranges":[{"sheetId":id_camp,"startRowIndex":1,"endRowIndex":500,"startColumnIndex":8,"endColumnIndex":9}],"booleanRule":{"condition":{"type":"TEXT_CONTAINS","values":[{"userEnteredValue":"Bloqueado"}]},"format":{"backgroundColor":hex_rgb("#FEE2E2"),"textFormat":{"foregroundColor":hex_rgb("#991B1B"),"bold":True}}}},"index":1}},
    {"addConditionalFormatRule":{"rule":{"ranges":[{"sheetId":id_camp,"startRowIndex":1,"endRowIndex":500,"startColumnIndex":8,"endColumnIndex":9}],"booleanRule":{"condition":{"type":"TEXT_CONTAINS","values":[{"userEnteredValue":"andamento"}]},"format":{"backgroundColor":hex_rgb("#FEF3C7"),"textFormat":{"foregroundColor":hex_rgb("#92400E")}}}},"index":2}},
] + [{"updateDimensionProperties":{"range":{"sheetId":id_camp,"dimension":"COLUMNS","startIndex":i,"endIndex":i+1},"properties":{"pixelSize":w},"fields":"pixelSize"}} for i,w in enumerate(widths)]
svc.spreadsheets().batchUpdate(spreadsheetId=SPREADSHEET_ID, body={"requests":reqs}).execute()
print("✅ Aba CAMPANHAS configurada")

# --- CLIENTES ---
h_cli = ["Slug","Nome Completo","Email Operador","Email Copy","Email Arte","Email Dados","Email Aprovador"]
rows_cli = [[c["slug"],c["name"],c["operador_email"],c["copy_email"],c["criacao_email"],"",c["approver_email"]] for c in CLIENTES]
reqs_cli = [
    {"updateSheetProperties":{"properties":{"sheetId":id_cli,"gridProperties":{"frozenRowCount":1}},"fields":"gridProperties.frozenRowCount"}},
    {"updateCells":{"range":{"sheetId":id_cli,"startRowIndex":0,"endRowIndex":1,"startColumnIndex":0,"endColumnIndex":len(h_cli)},"rows":[{"values":[{"userEnteredValue":{"stringValue":h},"userEnteredFormat":header_fmt()} for h in h_cli]}],"fields":"userEnteredValue,userEnteredFormat"}},
    {"updateCells":{"range":{"sheetId":id_cli,"startRowIndex":1,"endRowIndex":1+len(rows_cli),"startColumnIndex":0,"endColumnIndex":len(h_cli)},"rows":[{"values":[{"userEnteredValue":{"stringValue":str(v)}} for v in row]} for row in rows_cli],"fields":"userEnteredValue"}},
] + [{"updateDimensionProperties":{"range":{"sheetId":id_cli,"dimension":"COLUMNS","startIndex":i,"endIndex":i+1},"properties":{"pixelSize":w},"fields":"pixelSize"}} for i,w in enumerate([110,200,240,230,230,230,230])]
svc.spreadsheets().batchUpdate(spreadsheetId=SPREADSHEET_ID, body={"requests":reqs_cli}).execute()
print("✅ Aba CLIENTES configurada")

# --- LINKS CALENDÁRIOS ---
h_lnk = ["Cliente (slug)","Nome Completo","Link Calendário HTML","Link Pasta Drive","Observações"]
rows_lnk = [[c["slug"],c["name"],"","",""] for c in CLIENTES]
reqs_lnk = [
    {"updateSheetProperties":{"properties":{"sheetId":id_lnk,"gridProperties":{"frozenRowCount":1}},"fields":"gridProperties.frozenRowCount"}},
    {"updateCells":{"range":{"sheetId":id_lnk,"startRowIndex":0,"endRowIndex":1,"startColumnIndex":0,"endColumnIndex":len(h_lnk)},"rows":[{"values":[{"userEnteredValue":{"stringValue":h},"userEnteredFormat":header_fmt()} for h in h_lnk]}],"fields":"userEnteredValue,userEnteredFormat"}},
    {"updateCells":{"range":{"sheetId":id_lnk,"startRowIndex":1,"endRowIndex":1+len(rows_lnk),"startColumnIndex":0,"endColumnIndex":len(h_lnk)},"rows":[{"values":[{"userEnteredValue":{"stringValue":str(v)}} for v in row]} for row in rows_lnk],"fields":"userEnteredValue"}},
] + [{"updateDimensionProperties":{"range":{"sheetId":id_lnk,"dimension":"COLUMNS","startIndex":i,"endIndex":i+1},"properties":{"pixelSize":w},"fields":"pixelSize"}} for i,w in enumerate([120,200,380,300,200])]
svc.spreadsheets().batchUpdate(spreadsheetId=SPREADSHEET_ID, body={"requests":reqs_lnk}).execute()
print("✅ Aba LINKS CALENDÁRIOS configurada")
print(f"\n🔗 Planilha: https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/edit")

In [ ]:
# ━━━ CÉLULA 5 — Sobe o Apps Script e publica o webapp ━━━━━━━━━━━━━━━━━━━━━━
import re, requests as rq
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request as GRequest

creds = Credentials.from_authorized_user_file("/tmp/google_token.json", SCOPES)
creds.refresh(GRequest())
hdrs = {"Authorization": f"Bearer {creds.token}", "Content-Type": "application/json"}

SCRIPT_API = "https://script.googleapis.com/v1"

# Conteúdo dos arquivos Apps Script
# (versão inline — cópia dos arquivos do repositório)
GS_FILES = [
  {"name":"appsscript","type":"JSON","source":'{"timeZone":"America/Sao_Paulo","dependencies":{},"exceptionLogging":"STACKDRIVER","runtimeVersion":"V8"}'},
]

# Lê os arquivos do repositório clonado no Colab (se disponível)
import os
gs_dir = "/content/apps_script"
if not os.path.exists(gs_dir):
    # Faz download direto do GitHub
    import urllib.request
    os.makedirs(gs_dir, exist_ok=True)
    base = "https://raw.githubusercontent.com/nataliacvmb-maker/rotina-nfp-afesu/main/scripts/apps_script"
    for fname in ["Codigo.gs","Email.gs","Calendario.gs","Setup.gs","Calendario.html"]:
        urllib.request.urlretrieve(f"{base}/{fname}", f"{gs_dir}/{fname}")
    print("Arquivos baixados do GitHub")

for name, ftype in [("Codigo","SERVER_JS"),("Email","SERVER_JS"),("Calendario","SERVER_JS"),("Setup","SERVER_JS"),("Calendario","HTML")]:
    ext = ".gs" if ftype == "SERVER_JS" else ".html"
    path = f"{gs_dir}/{name}{ext}"
    if os.path.exists(path):
        GS_FILES.append({"name": name, "type": ftype, "source": open(path).read()})

# Cria projeto Apps Script
r = rq.post(f"{SCRIPT_API}/projects", headers=hdrs, json={"title":"Calendário NFP — Comunicação", "parentId": SPREADSHEET_ID})
if r.status_code == 200:
    script_id = r.json()["scriptId"]
    print(f"✅ Projeto criado: {script_id}")
else:
    m = re.search(r'"scriptId"\s*:\s*"([^"]+)"', r.text)
    script_id = m.group(1) if m else None
    print(f"ℹ Projeto existente: {script_id}")

if script_id:
    r2 = rq.put(f"{SCRIPT_API}/projects/{script_id}/content", headers=hdrs, json={"files": GS_FILES})
    if r2.status_code == 200:
        print(f"✅ {len(GS_FILES)} arquivos enviados")
    else:
        print(f"⚠ Erro upload: {r2.status_code} — {r2.text[:300]}")

    # Publica webapp
    r3 = rq.post(f"{SCRIPT_API}/projects/{script_id}/deployments", headers=hdrs, json={"deploymentConfig":{"scriptId":script_id,"manifestFileName":"appsscript","description":"Webapp calendário NFP"}})
    if r3.status_code == 200:
        d = r3.json()
        for ep in d.get("entryPoints",[]):
            if ep.get("entryPointType") == "WEB_APP":
                webapp_url = ep["webAppEntryPoint"]["url"]
                print(f"✅ Webapp: {webapp_url}")
                # Preenche LINKS CALENDÁRIOS
                updates = [{"range":f"LINKS CALENDÁRIOS!C{i+2}","values":[[f"{webapp_url}?cliente={c['slug']}"]]} for i,c in enumerate(CLIENTES)]
                svc.spreadsheets().values().batchUpdate(spreadsheetId=SPREADSHEET_ID, body={"valueInputOption":"USER_ENTERED","data":updates}).execute()
                print("✅ URLs de calendário gravadas na planilha")
    else:
        print(f"⚠ Erro deploy: {r3.status_code} — {r3.text[:300]}")
        print("→ Publique manualmente: Apps Script → Implantar → Nova implantação → Aplicativo da Web")

print(f"\n🎉 Tudo pronto!")
print(f"   Planilha: https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/edit")
if script_id:
    print(f"   Apps Script: https://script.google.com/d/{script_id}/edit")